# Large Assignment 02 - Image Dataset

## 1. Setup, EDA-Informed Preprocessing, and Feature Extraction (ResNet-18)

In [1]:
# 1. Setup, EDA-Informed Preprocessing, and Feature Extraction (ResNet-18)
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import numpy as np
import os
import json

# 1. Device Setup for M2 Max
# Using MPS (Metal Performance Shaders) for 30 GPU cores
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Preprocessing Informed by previous EDA
# Values sourced directly from EDA outputs [Section 4]
mean = [0.4416, 0.4461, 0.4718] 
std = [0.2040, 0.2081, 0.2058]

# Training transforms include augmentation for cluttered backgrounds
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
    # Note: HorizontalFlip is excluded per EDA findings on 6/9 and 2/5 ambiguity
])

# Test transforms only include normalization
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# 3. Load Datasets
train_set = datasets.SVHN(root='./data', split='train', download=True, transform=train_transform)
test_set = datasets.SVHN(root='./data', split='test', download=True, transform=test_transform)

# Recommended batch size for M2 Max with 32GB Ram
train_loader = DataLoader(train_set, batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False, num_workers=0)

def extract_features(dataloader, model):
    model.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs = inputs.to(device)
            # Forward pass through frozen backbone
            features = model(inputs)
            # Flatten features to 1D vector
            features = features.view(features.size(0), -1)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            
            if (batch_idx + 1) % 50 == 0:
                print(f"Processed batch {batch_idx + 1}/{len(dataloader)}")
    return np.concatenate(all_features), np.concatenate(all_labels)

# Initialize ResNet-18 as a feature extractor
print("Initializing ResNet-18 for feature extraction...")
resnet18 = models.resnet18(weights='DEFAULT')
# Remove the final fully connected layer to get raw features
resnet18 = nn.Sequential(*list(resnet18.children())[:-1])
resnet18 = resnet18.to(device)

# Execute Extraction
print("Extracting training features...")
X_train_features, y_train = extract_features(train_loader, resnet18)
print("Extracting test features...")
X_test_features, y_test = extract_features(test_loader, resnet18)

# Save to disk as planned
os.makedirs('features', exist_ok=True)
np.save('features/resnet18_train_32.npy', X_train_features)
np.save('features/resnet18_test_32.npy', X_test_features)
np.save('features/train_labels.npy', y_train)
np.save('features/test_labels.npy', y_test)

print(f"Features saved! Shape: {X_train_features.shape}")

Using device: mps
Initializing ResNet-18 for feature extraction...
Extracting training features...
Processed batch 50/287
Processed batch 100/287
Processed batch 150/287
Processed batch 200/287
Processed batch 250/287
Extracting test features...
Processed batch 50/102
Processed batch 100/102
Features saved! Shape: (73257, 512)


## 2. Classical ML on Pretrained Features

In [4]:
# 2. Classical ML on Pretrained Features
import time
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# 1. Load the features saved in Block 1
X_train = np.load('features/resnet18_train_32.npy')
X_test = np.load('features/resnet18_test_32.npy')
y_train = np.load('features/train_labels.npy')
y_test = np.load('features/test_labels.npy')

# 2. Define the "Tournament" of models
# Note: use 'balance' weights to address the 3.03x imbalance found in the EDA
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Linear SVM": LinearSVC(class_weight='balanced', max_iter=2000),
    "Random Forest": RandomForestClassifier(n_estimators=100, n_jobs=-1),
    "MLP (Neural Baseline)": MLPClassifier(hidden_layer_sizes=(256,), max_iter=20),
    "Gaussian Naive Bayes": GaussianNB()
}

ml_results = []

print(f"{'Model':<25} | {'Accuracy':<10} | {'Macro F1':<10} | {'Time (s)':<10}")
print("-" * 65)

# 3. Train and Evaluate each model
for name, clf in classifiers.items():
    start_time = time.time()
    
    # Train the model
    clf.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Make predictions
    y_pred = clf.predict(X_test)
    
    # Calculate metrics (Macro F1 is critical due to class imbalance)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    cm = confusion_matrix(y_test, y_pred).tolist()
    
    print(f"{name:<25} | {acc:<10.4f} | {f1:<10.4f} | {train_time:<10.2f}")
    
    # Store for JSON export later
    ml_results.append({
        "model_name": name,
        "accuracy": float(acc),
        "macro_f1": float(f1),
        "train_time_s": float(train_time),
        "confusion_matrix": cm
    })

# 4. Save results for your webpage
with open('ml_section1_results_32.json', 'w') as f:
    json.dump(ml_results, f, indent=2)

Model                     | Accuracy   | Macro F1   | Time (s)  
-----------------------------------------------------------------


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression       | 0.4264     | 0.4236     | 39.04     
Linear SVM                | 0.4345     | 0.4277     | 180.91    
Random Forest             | 0.3504     | 0.3227     | 7.22      


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (20) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP (Neural Baseline)     | 0.4556     | 0.4407     | 5.96      
Gaussian Naive Bayes      | 0.2419     | 0.2421     | 0.10      


## 1.5 Feature Extraction: ResNet-18 with 224x224 Resize (Resolution Fix)

In [5]:
# 1.5 Setup, EDA-Informed Preprocessing, and Feature Extraction (ResNet-18) - Fixed Feature Extraction (224×224)
from torchvision import models as tv_models

# 1. Device Setup for M2 Max
# Using MPS (Metal Performance Shaders) for 30 GPU cores
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Preprocessing Informed by previous EDA
# Values sourced directly from EDA outputs [Section 4]
mean = [0.4416, 0.4461, 0.4718] 
std = [0.2040, 0.2081, 0.2058]

# Change 1: One unified transform for both train and test
extract_transform = transforms.Compose([
    transforms.Resize(224),          # ← THE KEY FIX: ResNet-18 expects 224×224
    transforms.ToTensor(),
    transforms.Normalize(mean, std)  # same EDA values as before
    # No augmentation — feature extraction needs deterministic, stable outputs
])

# 3. Load Datasets
# Change 2: Both train and test use the same extract_transform
train_set_feat = datasets.SVHN(root='./data', split='train', download=False, transform=extract_transform)
test_set_feat  = datasets.SVHN(root='./data', split='test',  download=False, transform=extract_transform)


train_loader_feat = DataLoader(train_set_feat, batch_size=128, shuffle=False, num_workers=0)
#                                              ↑ reduced from 256 because 224×224 images are 
#                                                49× larger in memory than 32×32

test_loader_feat = DataLoader(test_set_feat, batch_size=128, shuffle=False, num_workers=0)


def extract_features(dataloader, model):
    model.eval()
    all_features = []
    all_labels = []
    
    with torch.no_grad():
        for batch_idx, (inputs, labels) in enumerate(dataloader):
            inputs = inputs.to(device)
            # Forward pass through frozen backbone
            features = model(inputs)
            # Flatten features to 1D vector
            features = features.view(features.size(0), -1)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
            
            if (batch_idx + 1) % 50 == 0:
                print(f"Processed batch {batch_idx + 1}/{len(dataloader)}")
    return np.concatenate(all_features), np.concatenate(all_labels)

# Initialize ResNet-18 as a feature extractor
print("Initializing ResNet-18 for feature extraction...")
resnet18 = tv_models.resnet18(weights='DEFAULT')
# Remove the final fully connected layer to get raw features
resnet18 = nn.Sequential(*list(resnet18.children())[:-1])
resnet18 = resnet18.to(device)

# Execute Extraction
# Change 3: Save to different filenames so Block 1 files are preserved
print("Extracting training features...")
X_train_features, y_train = extract_features(train_loader_feat, resnet18)
print("Extracting test features...")
X_test_features, y_test   = extract_features(test_loader_feat,  resnet18)

# Save to disk as planned
os.makedirs('features', exist_ok=True)
np.save('features/resnet18_train_224.npy', X_train_features)  # ← different filename
np.save('features/resnet18_test_224.npy',  X_test_features)
np.save('features/train_labels.npy', y_train)   # labels don't change, can overwrite
np.save('features/test_labels.npy',  y_test)

print(f"Features saved! Shape: {X_train_features.shape}")
# Should still print (73257, 512) — same 512 features, but now properly computed

Using device: mps
Initializing ResNet-18 for feature extraction...
Extracting training features...
Processed batch 50/573
Processed batch 100/573
Processed batch 150/573
Processed batch 200/573
Processed batch 250/573
Processed batch 300/573
Processed batch 350/573
Processed batch 400/573
Processed batch 450/573
Processed batch 500/573
Processed batch 550/573
Extracting test features...
Processed batch 50/204
Processed batch 100/204
Processed batch 150/204
Processed batch 200/204
Features saved! Shape: (73257, 512)


## 2.5 Classical ML on 224x224 Features (Improve Results)

In [6]:
# 2.5 - Classical ML on Fixed Features (224×224)

# 1. Load the features saved in Block 1
# Change 1: Load the new 224×224 feature files
X_train = np.load('features/resnet18_train_224.npy')  # ← _224 files
X_test  = np.load('features/resnet18_test_224.npy')

y_train = np.load('features/train_labels.npy')
y_test = np.load('features/test_labels.npy')

# 2. Define the "Tournament" of models
# Note: use 'balance' weights to address the 3.03x imbalance found in the EDA
# Change 2: Fix MLP convergence
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Linear SVM":          LinearSVC(class_weight='balanced', max_iter=2000),
    "Random Forest":       RandomForestClassifier(n_estimators=100, n_jobs=-1),
    "MLP (Neural Baseline)": MLPClassifier(hidden_layer_sizes=(256,), max_iter=200), # ← 20→200
    "Gaussian Naive Bayes":  GaussianNB()
}


ml_results = []

print(f"{'Model':<25} | {'Accuracy':<10} | {'Macro F1':<10} | {'Time (s)':<10}")
print("-" * 65)

# 3. Train and Evaluate each model
for name, clf in classifiers.items():
    start_time = time.time()
    
    # Train the model
    clf.fit(X_train, y_train)
    train_time = time.time() - start_time
    
    # Make predictions
    y_pred = clf.predict(X_test)
    
    # Calculate metrics (Macro F1 is critical due to class imbalance)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    cm = confusion_matrix(y_test, y_pred).tolist()
    
    print(f"{name:<25} | {acc:<10.4f} | {f1:<10.4f} | {train_time:<10.2f}")
    
    # Store for JSON export later
    ml_results.append({
        "model_name": name,
        "accuracy": float(acc),
        "macro_f1": float(f1),
        "train_time_s": float(train_time),
        "confusion_matrix": cm
    })

# 4. Save results for your webpage
# Change 3: Save to a different JSON filename
with open('ml_section1_results_224.json', 'w') as f:  # ← _224
    json.dump(ml_results, f, indent=2)

Model                     | Accuracy   | Macro F1   | Time (s)  
-----------------------------------------------------------------


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression       | 0.6248     | 0.6127     | 38.01     
Linear SVM                | 0.6276     | 0.6121     | 128.13    
Random Forest             | 0.4756     | 0.4211     | 12.86     


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


MLP (Neural Baseline)     | 0.6044     | 0.5887     | 55.86     
Gaussian Naive Bayes      | 0.3972     | 0.3729     | 0.10      


## 3. Pipeline Comparison

In [8]:
# 3.1 Prepare Remaining Feature Sets (MobileNetV2 & Raw Pixels)
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.base import clone
import time

print("--- PART 1: PREPARING ADDITIONAL FEATURES ---")

# 1. Initialize MobileNetV2 (Using the tv_models alias from Block 1.5)
print("Initializing MobileNetV2 for feature extraction...")
mobilenet = tv_models.mobilenet_v2(weights='DEFAULT')
# MobileNetV2 has a specific structure; replacing the classifier with Identity gives the 1280-d pooled features
mobilenet.classifier = nn.Identity()
mobilenet = mobilenet.to(device)

# 2. Extract MobileNetV2 Features (using the 224x224 loaders from Block 1.5)
print("Extracting MobileNetV2 features (takes a few minutes on M2 Max)...")
X_train_mbv2, _ = extract_features(train_loader_feat, mobilenet)
X_test_mbv2, _ = extract_features(test_loader_feat, mobilenet)

np.save('features/mobilenet_train_224.npy', X_train_mbv2)
np.save('features/mobilenet_test_224.npy', X_test_mbv2)
print(f"MobileNetV2 Features saved! Shape: {X_train_mbv2.shape}")

# 3. Prepare Raw Pixels (using the 32x32 loaders from Block 1)
print("\nPreparing Raw Pixel baselines (32x32)...")
def extract_raw_pixels(dataloader):
    all_pixels = []
    for inputs, _ in dataloader:
        # Flatten the 3x32x32 images into 3072-d vectors
        flat = inputs.view(inputs.size(0), -1).numpy()
        all_pixels.append(flat)
    return np.concatenate(all_pixels)

X_train_raw = extract_raw_pixels(train_loader)
X_test_raw = extract_raw_pixels(test_loader)
print(f"Raw Pixels ready! Shape: {X_train_raw.shape}")


print("\n--- PART 2: THE 8-PIPELINE TOURNAMENT ---")

# Load ResNet features (already saved from Block 1.5)
X_train_rn18 = np.load('features/resnet18_train_224.npy')
X_test_rn18 = np.load('features/resnet18_test_224.npy')
y_train = np.load('features/train_labels.npy')
y_test = np.load('features/test_labels.npy')

# Base classifiers (with balanced weights based on your EDA)
lr_base = LogisticRegression(max_iter=1000, class_weight='balanced')
svm_base = LinearSVC(class_weight='balanced', max_iter=2000)
rf_base = RandomForestClassifier(n_estimators=100, n_jobs=-1)

# Define the 8 Pipelines
pipelines = [
    {"id": 1, "name": "ResNet18 -> None -> LR", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('clf', clone(lr_base))])},
    
    {"id": 2, "name": "ResNet18 -> PCA-128 -> LR", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(lr_base))])},
    
    {"id": 3, "name": "ResNet18 -> PCA-128 -> SVM", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(svm_base))])},
    
    {"id": 4, "name": "ResNet18 -> PCA-128 -> RF", "X_tr": X_train_rn18, "X_te": X_test_rn18,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(rf_base))])},
    
    {"id": 5, "name": "MobileNetV2 -> None -> LR", "X_tr": X_train_mbv2, "X_te": X_test_mbv2,
     "pipe": Pipeline([('clf', clone(lr_base))])},
    
    {"id": 6, "name": "MobileNetV2 -> PCA-256 -> LR", "X_tr": X_train_mbv2, "X_te": X_test_mbv2,
     "pipe": Pipeline([('pca', PCA(n_components=256)), ('clf', clone(lr_base))])},
    
    {"id": 7, "name": "Raw Pixels -> PCA-128 -> LR", "X_tr": X_train_raw, "X_te": X_test_raw,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(lr_base))])},
    
    {"id": 8, "name": "Raw Pixels -> PCA-128 -> SVM", "X_tr": X_train_raw, "X_te": X_test_raw,
     "pipe": Pipeline([('pca', PCA(n_components=128)), ('clf', clone(svm_base))])}
]

pipeline_results = []

print(f"{'ID':<3} | {'Pipeline Name':<30} | {'Accuracy':<8} | {'Macro F1':<8} | {'Train Time (s)'}")
print("-" * 75)

for p in pipelines:
    start_time = time.time()
    
    # Fit the pipeline (PCA + Classifier)
    p["pipe"].fit(p["X_tr"], y_train)
    train_time = time.time() - start_time
    
    # Predict
    y_pred = p["pipe"].predict(p["X_te"])
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    print(f"{p['id']:<3} | {p['name']:<30} | {acc:<8.4f} | {f1:<8.4f} | {train_time:<10.2f}")
    
    pipeline_results.append({
        "rank": 0, # Will sort in JS later on the webpage
        "pipeline": p["name"],
        "accuracy": float(acc),
        "macro_f1": float(f1),
        "train_time_s": float(train_time)
    })
    
# Save results to JSON for the webpage dashboard
with open('ml_section2_pipelines.json', 'w') as f:
    json.dump(pipeline_results, f, indent=2)

--- PART 1: PREPARING ADDITIONAL FEATURES ---
Initializing MobileNetV2 for feature extraction...
Extracting MobileNetV2 features (takes a few minutes on M2 Max)...
Processed batch 50/573
Processed batch 100/573
Processed batch 150/573
Processed batch 200/573
Processed batch 250/573
Processed batch 300/573
Processed batch 350/573
Processed batch 400/573
Processed batch 450/573
Processed batch 500/573
Processed batch 550/573
Processed batch 50/204
Processed batch 100/204
Processed batch 150/204
Processed batch 200/204
MobileNetV2 Features saved! Shape: (73257, 1280)

Preparing Raw Pixel baselines (32x32)...
Raw Pixels ready! Shape: (73257, 3072)

--- PART 2: THE 8-PIPELINE TOURNAMENT ---
ID  | Pipeline Name                  | Accuracy | Macro F1 | Train Time (s)
---------------------------------------------------------------------------


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


1   | ResNet18 -> None -> LR         | 0.6248   | 0.6127   | 45.21     
2   | ResNet18 -> PCA-128 -> LR      | 0.5665   | 0.5545   | 1.46      
3   | ResNet18 -> PCA-128 -> SVM     | 0.5701   | 0.5512   | 24.58     
4   | ResNet18 -> PCA-128 -> RF      | 0.4489   | 0.3913   | 8.26      


/Users/vnnguyen/Desktop/HK252/P4AIDS/EDA-Techniques-for-AI-and-DS/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


5   | MobileNetV2 -> None -> LR      | 0.5792   | 0.5600   | 62.83     
6   | MobileNetV2 -> PCA-256 -> LR   | 0.5369   | 0.5185   | 3.46      
7   | Raw Pixels -> PCA-128 -> LR    | 0.1681   | 0.1548   | 4.66      
8   | Raw Pixels -> PCA-128 -> SVM   | 0.1933   | 0.1636   | 38.82     


## 4. Fine-Tuning ResNet-18

In [10]:
# 4. Transfer Learning (Fine-Tuning)
import torch.optim as optim
from torch.optim import lr_scheduler
import copy
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

print("--- BLOCK 4: FINE-TUNING RESNET-18 ---")

# 1. Calculate Class Weights (from EDA Imbalance)
# Need to compute weights inversely proportional to class frequencies
# From the EDA: class counts for 0-9
class_counts = np.array([6692, 18960, 14734, 11379, 9981, 9266, 7704, 7614, 6705, 6254])
total_samples = class_counts.sum()
num_classes = len(class_counts)

# Calculate weights: total / (num_classes * count)
weights = total_samples / (num_classes * class_counts)
class_weights = torch.FloatTensor(weights).to(device)

print(f"Computed Class Weights: {weights}")

# 2. Initialize Model for Fine-Tuning
print("Initializing ResNet-18 for Fine-Tuning...")
finetune_model = tv_models.resnet18(weights='DEFAULT')

# Replace the final layer for 10 classes
num_ftrs = finetune_model.fc.in_features
finetune_model.fc = nn.Linear(num_ftrs, 10)
finetune_model = finetune_model.to(device)

# 3. Define Loss, Optimizer, and Scheduler
num_epochs = 10 
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Use a smaller learning rate since this is fine-tuning an already trained model
optimizer = optim.Adam(finetune_model.parameters(), lr=1e-4)

# Cosine Annealing for smooth LR decay
scheduler = lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

# 4. Training Loop
def train_model(model, criterion, optimizer, scheduler, num_epochs=10):
    start_time = time.time()
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': []
    }

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        print('-' * 10)

        # Each epoch has a training and validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()  
                dataloader = train_loader_feat # Using the 224x224 loaders
            else:
                model.eval()   
                dataloader = test_loader_feat

            running_loss = 0.0
            running_corrects = 0

            # Iterate over data
            for inputs, labels in dataloader:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Forward
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    # Backward + optimize only if in training phase
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(dataloader.dataset)
            epoch_acc = running_corrects.float() / len(dataloader.dataset)
            # .double() = float64 — not supported on Apple MPS
            # .float()  = float32 — the correct type for MPS

            print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')
            
            # Save history
            if phase == 'train':
                history['train_loss'].append(epoch_loss)
                history['train_acc'].append(float(epoch_acc))
            else:
                history['val_loss'].append(epoch_loss)
                history['val_acc'].append(float(epoch_acc))

            # Deep copy the model if it's the best validation accuracy
            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - start_time
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:4f}')
    
    # Load best model weights
    model.load_state_dict(best_model_wts)
    return model, history

# 5. Execute Training
print("Starting Fine-Tuning process...")
best_model, training_history = train_model(finetune_model, criterion, optimizer, scheduler, num_epochs=num_epochs)

# 6. Final Evaluation and JSON Export
print("Evaluating best model to generate final metrics...")
best_model.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for inputs, labels in test_loader_feat:
        inputs = inputs.to(device)
        outputs = best_model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

final_acc = accuracy_score(all_labels, all_preds)
final_f1 = f1_score(all_labels, all_preds, average='macro')
final_cm = confusion_matrix(all_labels, all_preds).tolist()

# Extract per-class F1 for the webpage dashboard
per_class_f1 = f1_score(all_labels, all_preds, average=None).tolist()

print(f"Final Fine-Tuned Accuracy: {final_acc:.4f}")
print(f"Final Fine-Tuned Macro F1: {final_f1:.4f}")

finetune_results = {
    "model": "ResNet-18 (Fine-Tuned)",
    "epochs": num_epochs,
    "final_accuracy": float(final_acc),
    "final_macro_f1": float(final_f1),
    "per_class_f1": per_class_f1,    # Added per your request
    "confusion_matrix": final_cm,
    "history": training_history
}

with open('ml_section3_finetune.json', 'w') as f:
    json.dump(finetune_results, f, indent=2)
    
print("Fine-tuning results saved to ml_section3_finetune.json!")

--- BLOCK 4: FINE-TUNING RESNET-18 ---
Computed Class Weights: [1.48369695 0.52367616 0.67387675 0.87256349 0.99478008 1.07154112
 1.28879803 1.30403205 1.48082028 1.58760793]
Initializing ResNet-18 for Fine-Tuning...
Starting Fine-Tuning process...
Epoch 1/10
----------
Train Loss: 0.3894 Acc: 0.8795
Val Loss: 0.1793 Acc: 0.9465

Epoch 2/10
----------
Train Loss: 0.1525 Acc: 0.9559
Val Loss: 0.1566 Acc: 0.9566

Epoch 3/10
----------
Train Loss: 0.0810 Acc: 0.9781
Val Loss: 0.1710 Acc: 0.9559

Epoch 4/10
----------
Train Loss: 0.0444 Acc: 0.9868
Val Loss: 0.1885 Acc: 0.9538

Epoch 5/10
----------
Train Loss: 0.0235 Acc: 0.9933
Val Loss: 0.1839 Acc: 0.9565

Epoch 6/10
----------
Train Loss: 0.0112 Acc: 0.9968
Val Loss: 0.1872 Acc: 0.9608

Epoch 7/10
----------
Train Loss: 0.0045 Acc: 0.9991
Val Loss: 0.1816 Acc: 0.9624

Epoch 8/10
----------
Train Loss: 0.0019 Acc: 0.9998
Val Loss: 0.1784 Acc: 0.9632

Epoch 9/10
----------
Train Loss: 0.0010 Acc: 1.0000
Val Loss: 0.1793 Acc: 0.9639

Epo